# Inflation Surprise

This model will look at trading energy futures based on their inflation surprise using Bloomberg Inflation Surprise Indices

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
from   statsmodels.regression.rolling import RollingOLS

In [ ]:
data_path = os.path.join(os.getcwd(), "data")

In [ ]:
inf_path = os.path.join(data_path, "InflationSurprise.parquet")
df_inf   = pd.read_parquet(path = inf_path, engine = "pyarrow")

In [ ]:
fut_path   = os.path.join(data_path, "FutPX.parquet")
df_fut_rtn = (pd.read_parquet(
    path = fut_path, engine = "pyarrow").
    pivot(index = "date", columns = "security", values = "PX_LAST").
    pct_change().
    reset_index().
    melt(id_vars = "date", value_name = "fut_rtn").
    dropna())

# Trading OLS

In [ ]:
df_inf_prep = (df_inf.pivot(
    index = "date", columns = "country", values = "value").
    diff().
    shift().
    reset_index().
    melt(id_vars = "date", value_name = "inf_surp").
    dropna())

In [ ]:
df_combined = (df_fut_rtn.pivot(
    index = "date", columns = "security", values = "fut_rtn").
    apply(lambda x: x * (0.1 / (x.ewm(span = 10, adjust = False).std() * np.sqrt(252)))).
    reset_index().
    melt(id_vars = "date", value_name = "vol_rtn").
    dropna().
    merge(right = df_inf_prep, how = "inner", on = ["date"]).
    assign(group_var = lambda x: x.security + " " + x.country))

In [ ]:
def _get_vol(df: pd.DataFrame) -> pd.DataFrame: 

    df_out = (df.assign(
        vol     = lambda x: x.fut_rtn.ewm(span = 100, adjust = False).std() * np.sqrt(252),
        lagged  = lambda x: (0.1 / x.vol) * x.fut_rtn,
        perfect = lambda x: (0.1 / x.vol.shift()) * x.fut_rtn).
        drop(columns = ["vol"]))

    return df_out

df_combined = (df_fut_rtn.merge(
    right = df_inf_prep, how = "inner", on = ["date"]).
    set_index("date").
    assign(group_var = lambda x: x.security + " " + x.country).
    groupby("group_var").
    apply(_get_vol, include_groups = False).
    reset_index())

In [ ]:
def _ols(df: pd.DataFrame) -> pd.DataFrame: 

    model = (sm.OLS(
        endog = df.value,
        exog  = sm.add_constant(df.inf_surp)).
        fit())

    df_param   = model.params.to_frame(name = "param_val").reset_index()
    df_pvalues = model.pvalues.to_frame(name = "pvalues").reset_index()
    df_tstats  = model.tvalues.to_frame(name = "tstat").reset_index()
    
    df_out = (df_param.merge(
        right = df_pvalues, how = "inner", on = ["index"]).
        merge(right = df_tstats, how = "inner", on = ["index"]))

    return df_out

display(df_combined.melt(
    id_vars = ["date", "group_var", "security", "country", "inf_surp"]).
    dropna().
    set_index("date").
    assign(group_var = lambda x: x.group_var + " " + x.variable).
    groupby("group_var").
    apply(_ols).
    reset_index().
    drop(columns = ["level_1"]).
    assign(
        str_tmp  = lambda x: x.group_var.str.split(" "),
        security = lambda x: x.str_tmp.str[0],
        country  = lambda x: x.str_tmp.str[1],
        vol      = lambda x: x.str_tmp.str[2]).
    drop(columns = ["group_var", "str_tmp"]).
    rename(columns = {"index": "param"}).
    melt(id_vars = ["param", "security", "country", "vol"]).
    rename(columns = {
        "country": "Country",
        "security": "Security",
        "param"   : "Param",
        "variable": "",
        "vol"     : "Vol. Target"}).
    replace({
        "param_val": "Param Val",
        "pvalues"  : "$p$-value",
        "tstat"    : "$t$-stat",
        "fut_rtn"  : "Raw",
        "lagged"   : "Lagged",
        "perfect"  : "Perfect",
        "const"    : r"$\alpha$",
        "inf_surp" : r"Inf Sur."}).
    pivot(index = ["Country", "Security", "Param"], columns = ["", "Vol. Target"], values = "value").
    apply(lambda x: np.round(x,3)))

In [ ]:
def _get_resid(df: pd.DataFrame, q: int = 10) -> pd.DataFrame: 
    
    df_out = (sm.OLS(
        endog = df.fut_rtn,
        exog  = sm.add_constant(df.inf_surp)).
        fit().
        resid.
        to_frame(name = "resid").
        assign(
            lag_resid = lambda x: x.resid.shift(),
            decile    = lambda x: pd.qcut(x = x.resid, q = q, labels = [i + 1 for i in range(q)]).shift()).
        merge(right = df, how = "inner", on = ["date"]))
    
    return df_out


df_resid = (df_fut_rtn.merge(
    right = df_inf_prep, how = "inner", on = ["date"]).
    set_index("date").
    assign(group_var = lambda x: x.security + " " + x.country).
    groupby("group_var").
    apply(_get_resid, include_groups = False).
    reset_index().
    assign(signal_rtn = lambda x: -np.sign(x.lag_resid) * x.fut_rtn))

In [ ]:
df_resid_wider = (df_resid.pivot(
    index = "date", columns = ["country", "security"], values = "signal_rtn"))

df_perf = (df_resid_wider.apply(
    lambda x: x * (0.1 / (x.ewm(span = 100, adjust = False).std() * np.sqrt(252)))).
    reset_index().
    melt(id_vars = [("date", "")], value_name = "rtn").
    rename(columns = {("date", ""): "date"}).
    assign(group = "perfect"))

df_lagged = (df_resid_wider.apply(
    lambda x: x * (0.1 / (x.ewm(span = 100, adjust = False).std().shift() * np.sqrt(252)))).
    apply(lambda x: np.where(np.abs(x) == np.inf, np.nan, x)).
    reset_index().
    melt(id_vars = [("date", "")], value_name = "rtn").
    rename(columns = {("date", ""): "date"}).
    assign(group = "lagged"))

df_resid_combined = (pd.concat([
    df_perf, df_lagged]).
    assign(group_var = lambda x: x.country + " " + x.group).
    dropna())

In [ ]:
group_vars = df_resid_combined.group_var.drop_duplicates().sort_values().to_list()
fig, axes  = plt.subplots(ncols = len(group_vars) // 2, nrows = len(group_vars) // 2, figsize = (20,10))

for group_var, ax in zip(group_vars, axes.flatten()): 
    
    (df_resid_combined.query(
        "group_var == @group_var").
        rename(columns = {"security": ""}).
        pivot(index = "date", columns = "", values = "rtn").
        apply(lambda x: np.cumprod(1 + x)).
        plot(
            logy = True,
            ax   = ax,
            ylabel = "Cuml. Ret. Log-Scaled",
            title  = group_var.capitalize().replace("Us", "US").replace("Uk", "UK") + " Volatility"))

fig.suptitle("Cumulative Returns of Trading Residuals of Inflation Surprise\n10% Volatility Targeting From {} to {}".format(
    df_resid_combined.date.min().date(),
    df_resid_combined.date.max().date()))

plt.tight_layout()

fig.savefig(os.path.join(os.getcwd(), "LaTex", "InflationSurpriseCumlReturns.."))

In [ ]:
(df_resid_combined.drop(
    columns = ["date", "group_var"]).
    groupby(["security", "group", "country"]).
    agg(lambda x: x.mean() / x.std() * np.sqrt(252)).
    reset_index().
    replace({
        "lagged" : "Lagged",
        "perfect": "Perfect"}).
    rename(columns = {
        "group": "Vol Target"}).
    pivot(index = ["country", "security"], columns = "Vol Target", values = "rtn").
    apply(lambda x: np.round(x,3)))

# Trading OLS Deciled Residuals

In [ ]:
df_decile_sharpe = (df_resid[
    ["security", "decile", "country", "fut_rtn"]].
    groupby(["security", "decile", "country"]).
    agg(lambda x: x.mean() / x.std() * np.sqrt(252)).
    reset_index())

In [ ]:
df_tmp = (df_decile_sharpe.assign(
    group_var = lambda x: x.security + " " + x.country))

group_vars = df_tmp.group_var.drop_duplicates().sort_values().to_list()
fig, axes  = plt.subplots(ncols = 4, nrows = 3, figsize = (20,12))

for group_var, ax in zip(group_vars, axes.flatten()): 
    
    (df_tmp.query(
        "group_var == @group_var").
        set_index("decile")
        [["fut_rtn"]].
        plot(
            kind   = "bar",
            legend = False,
            xlabel = "",
            rot    = 0,
            ylabel = "Annualized Sharpe",
            title  = group_var,
            ax     = ax))

fig.suptitle("Sharpe per signal decile")
plt.tight_layout()

In [ ]:
df_decile_group = (df_tmp.query(
    "decile == [1,2,9,10]").
    assign(a = lambda x: np.where(x.decile <= 2, "lower_group", "upper_group")))

In [ ]:
df_signal_rtn = (df_decile_group[
    ["group_var", "a", "fut_rtn"]].
    groupby(["group_var", "a"]).
    agg("prod").
    reset_index().
    assign(scaler = lambda x: np.where(x.fut_rtn > 0, 1, 0)).
    drop(columns = ["fut_rtn"]).
    merge(right = df_decile_group, how = "inner", on = ["group_var", "a"]).
    rename(columns = {"fut_rtn": "sharpe"}).
    merge(right = df_resid, how = "outer", on = ["group_var", "security", "decile", "country"]))

In [ ]:
df_perf = (df_signal_rtn.assign(
    signal_rtn = lambda x: np.sign(x.sharpe * x.scaler) * x.fut_rtn).
    pivot(index = "date", columns = ["country", "security"], values = "signal_rtn").
    fillna(0).
    apply(lambda x: x * (0.1 / (x.ewm(span = 100, adjust = False).std() * np.sqrt(252)))).
    reset_index().
    melt(id_vars = [("date", "")], value_name = "signal_rtn").
    assign(group = "perf"))

df_lagged = (df_signal_rtn.assign(
    signal_rtn = lambda x: np.sign(x.sharpe * x.scaler) * x.fut_rtn).
    pivot(index = "date", columns = ["country", "security"], values = "signal_rtn").
    fillna(0).
    apply(lambda x: x * (0.1 / (x.ewm(span = 100, adjust = False).std().shift() * np.sqrt(252)))).
    apply(lambda x: np.where(np.abs(x) == np.inf, np.nan, x)).
    reset_index().
    melt(id_vars = [("date", "")], value_name = "signal_rtn").
    assign(group = "lagged"))

df_combined = (pd.concat([
    df_perf, df_lagged]).
    dropna().
    rename(columns = {("date", ""): "date"}))

In [ ]:
df_plot = (df_combined.assign(
    group_var = lambda x: x.group + " " + x.country))

group_vars = df_plot.group_var.drop_duplicates().sort_values().to_list()
fig, axes  = plt.subplots(ncols = 2, nrows = 2, figsize = (20,10))

for group_var, ax in zip(group_vars, axes.flatten()): 

    (df_plot.query(
        "group_var == @group_var").
        rename(columns = {"security": ""}).
        pivot(index = "date", columns = "", values = "signal_rtn").
        apply(lambda x: np.cumprod(1 + x)).
        plot(
            ax     = ax,
            logy   = True,
            ylabel = "Cuml. Ret. Log-Scaled",
            title  = group_var))

fig.suptitle("Trading outer deciles of OLS Residuals\n10% Volatility Targeting From {} to {}".format(
    df_plot.date.min().date(),
    df_plot.date.max().date()))
plt.tight_layout()

In [ ]:
(df_combined.assign(
    adj_rtn = lambda x: np.where(x.signal_rtn != 0, x.signal_rtn, np.nan)).
    melt(id_vars = ["date", "country", "security", "group"]).
    dropna().
    drop(columns = ["date"]).
    groupby(["country", "security", "group", "variable"]).
    agg(lambda x: x.mean() / x.std() * np.sqrt(252)).
    reset_index().
    rename(columns = {
        "group"   : "Vol Target",
        "variable": ""}).
    replace({
        "lagged"    : "Lagged",
        "perf"      : "Perfect",
        "adj_rtn"   : "Adj Sharpe",
        "signal_rtn": "Raw Sharpe"}).
    pivot(index = ["country", "security"], columns = ["Vol Target", ""], values = "value").
    apply(lambda x: np.round(x,3)))

In [ ]:
df_plot = (df_combined.assign(
    group_var = lambda x: x.group + " " + x.country))

group_vars = df_plot.group_var.drop_duplicates().sort_values().to_list()
fig, axes  = plt.subplots(ncols = 2, nrows = 2, figsize = (15,8))

for group_var, ax in zip(group_vars, axes.flatten()): 

    df_tmp = (df_plot.query(
        "group_var == @group_var").
        rename(columns = {"security": ""}).
        pivot(index = "date", columns = "", values = "signal_rtn").
        corr())

    sns.heatmap(
        data = df_tmp,
        ax   = ax,
        annot = True,
        cbar  = False)

fig.suptitle("Correlation of trading outer deciles of OLS Residuals\n10% Volatility Targeting From {} to {}".format(
    df_plot.date.min().date(),
    df_plot.date.max().date()))

In [ ]:
df_avg_corr = (df_combined.pivot(
    index = ["date", "group", "country"], columns = "security", values = "signal_rtn").
    reset_index().
    drop(columns = ["date"]).
    groupby(["group", "country"]).
    agg("corr").
    reset_index().
    rename(columns = {"security": "security1"}).
    melt(id_vars = ["group", "country", "security1"]).
    query("security1 != security").
    drop(columns = ["security1", "security"]).
    groupby(["group", "country"]).
    agg("mean").
    reset_index())

In [ ]:
df_tmp_lag = (df_fut_rtn.pivot(
    index = "date", columns = "security", values = "fut_rtn").
    apply(lambda x: x * (0.1 / (x.ewm(span = 100, adjust = False).std().shift() * np.sqrt(252)))).
    reset_index().
    melt(id_vars = "date").
    assign(group = "lagged"))

df_tmp_perf = (df_fut_rtn.pivot(
    index = "date", columns = "security", values = "fut_rtn").
    apply(lambda x: x * (0.1 / (x.ewm(span = 100, adjust = False).std() * np.sqrt(252)))).
    reset_index().
    melt(id_vars = "date").
    assign(group = "perf"))

df_und = (pd.concat([
    df_tmp_lag, df_tmp_perf]).
    dropna().
    pivot(index = ["date", "group"], columns = "security", values = "value").
    reset_index().
    drop(columns = ["date"]).
    groupby("group").
    apply(lambda x: x.corr()).
    reset_index().
    rename(columns = {"security": "security1"}).
    melt(id_vars = ["group", "security1"]).
    query("security1 != security").
    drop(columns = ["security1", "security"]).
    groupby(["group"]).
    agg("mean").
    reset_index().
    assign(country = "und"))

In [ ]:
(pd.concat([
    df_avg_corr, df_und]).
    replace({
        "lagged": "Lagged",
        "perf"  : "Perfect"}).
    pivot(index = "group", columns = "country", values = "value").
    apply(lambda x: np.round(x,3)).
    rename(columns = {"und": "Average"}))

In [ ]:
df_avg = (df_combined.drop(
    columns = ["security"]).
    groupby(["date", "country", "group"]).
    agg("mean").
    reset_index())

In [ ]:
(df_avg.assign(
    group = lambda x: np.where(x.group == "lagged", "Lagged", "Perfect"),
    tmp   = lambda x: x.group + " " + x.country).
    rename(columns = {"tmp": ""}).
    pivot(index = "date", columns = "", values = "signal_rtn").
    dropna().
    apply(lambda x: np.cumprod(1 + x)).
    plot(
        figsize = (8,5),
        logy    = True,
        ylabel  = "Cuml. Ret. Log-Scaled",
        xlabel  = "",
        title   = "Cumulative Returns of Trading Decile Residuals\n10% Volatility Targeting from {} to {}".format(
            df_avg.date.min().date(),
            df_avg.date.max().date())))

plt.tight_layout()

# Trading OLS Cross-Sectionally

In [ ]:
def _get_leg(df: pd.DataFrame) -> pd.DataFrame: 

    try:
    
        df_out = (df.assign(
            group = lambda x: pd.qcut(x = x.lag_resid, q = 2, labels = ["lower_group", "upper_group"])))

        return df_out

    except:
        pass

df_group = (df_resid.drop(
    columns = ["group_var", "decile"]).
    dropna().
    assign(group_var = lambda x: x.date.astype(str) + " " + x.country).
    groupby("group_var").
    apply(_get_leg, include_groups = False).
    reset_index(drop = True))

In [ ]:
df_fut_wider = (df_fut_rtn.pivot(
    index = "date", columns = "security", values = "fut_rtn"))

df_perf = (df_fut_wider.apply(
    lambda x: x * (0.1 / (x.ewm(span = 100, adjust = False).std() * np.sqrt(252)))).
    reset_index().
    melt(id_vars = "date", value_name = "value").
    assign(vol_group = "perf"))


df_lagged = (df_fut_wider.apply(
    lambda x: x * (0.1 / (x.ewm(span = 100, adjust = False).std().shift() * np.sqrt(252)))).
    reset_index().
    melt(id_vars = "date", value_name = "value").
    assign(vol_group = "lagged"))

In [ ]:
df_leg_rtn = (pd.concat([
    df_lagged, df_perf]).
    dropna().
    merge(right = df_group, how = "inner", on = ["date", "security"])
    [["date", "vol_group", "country", "group", "value"]].
    groupby(["date", "vol_group", "country", "group"]).
    agg("mean").
    reset_index())

In [ ]:
df_tmp = (df_leg_rtn.assign(
    group_var = lambda x: x.country + " " + x.vol_group))

group_vars = df_tmp.group_var.drop_duplicates().sort_values().to_list()
fig, axes  = plt.subplots(ncols = len(group_vars) // 2, nrows = len(group_vars) // 2, figsize = (20,8))

for group_var, ax in zip(group_vars, axes.flatten()): 

    (df_tmp.query(
        "group_var == @group_var").
        rename(columns = {"group": ""}).
        pivot(index = "date", columns = "", values = "value").
        apply(lambda x: np.cumprod(1 + x)).
        rename(columns = {
            "lower_group": "Group1",
            "upper_group": "Group2"}).
        plot(
            ax     = ax,
            logy   = True,
            ylabel = "Cuml. Ret. Log-Scaled",
            title  = group_var.replace("lagged", "Lagged").replace("perf", "Perfect") + " Volatility",
            xlabel = ""))

fig.suptitle("Cumulative Returns of Trading Long-Short Legs of OLS Residual\n10% Volatility Targeting from {} to {}".format(
    df_tmp.date.min().date(),
    df_tmp.date.max().date()))
plt.tight_layout()

In [ ]:
(df_leg_rtn.drop(
    columns = ["date"]).
    groupby(["vol_group", "country", "group"]).
    agg(lambda x: x.mean() / x.std() * np.sqrt(252)).
    reset_index().
    assign(group = lambda x: x.group.astype(str)).
    replace({
        "lagged"     : "Lagged",
        "perf"       : "Perfect",
        "lower_group": "Group1",
        "upper_group": "Group2"}).
    rename(columns = {
        "vol_group": "Vol. Target",
        "group"    : ""}).
    pivot(index = ["country", "Vol. Target"], columns = "", values = "value").
    apply(lambda x: np.round(x,3)))

In [ ]:
df_ls_rtn = (df_leg_rtn.pivot(
    index = ["date", "vol_group", "country"], columns = "group", values = "value").
    assign(spread = lambda x: x.lower_group - x.upper_group).
    reset_index())

In [ ]:
group_vars = df_ls_rtn.vol_group.drop_duplicates().sort_values().to_list()
fig, axes  = plt.subplots(ncols = len(group_vars), figsize = (20,6))

for group_var, ax in zip(group_vars, axes.flatten()): 

    (df_ls_rtn.query(
        "vol_group == @group_var").
        rename(columns = {"country": ""}).
        pivot(index = "date", columns = "", values = "spread").
        dropna().
        apply(lambda x: np.cumprod(1 + x)).
        plot(
            logy   = True,
            ax     = ax,
            ylabel = "Cuml. Ret. Log-Scaled",
            title  = group_var.capitalize().replace("Perf", "Perfect") + " Volatility"))

fig.suptitle("Cumulative Returns of Trading Long-Short Portfolios\n10% Volatility Target From {} to {}".format(
    df_ls_rtn.date.min().date(),
    df_ls_rtn.date.max().date()))
plt.tight_layout()

In [ ]:
(df_ls_rtn.drop(
    columns = ["date", "lower_group", "upper_group"]).
    groupby(["vol_group", "country"]).
    agg(lambda x: x.mean() / x.std() * np.sqrt(252)).
    reset_index().
    rename(columns = {"vol_group": ""}).
    pivot(index = "country", columns = "", values = "spread").
    rename(columns = {
        "lagged": "Lagged",
        "perf"  : "Perfect"}).
    apply(lambda x: np.round(x,3)))